In [1]:
import datetime as dt
from dotenv import load_dotenv
import pandas as pd
from backtesting import Strategy, Backtest
from backtesting.lib import crossover
from securities_load.securities.postgresql_database_functions import sqlalchemy_engine
from securities_load.securities.securities_table_functions import retrieve_ohlcv_from_to

ImportError: cannot import name 'Strategy' from 'backtesting' (unknown location)

In [ ]:
def localData(exchange, ticker, years):
    load_dotenv()
    engine = sqlalchemy_engine()
    exchange_code = "XNAS"
    ticker = "GOOG"
    num_of_years = 10
    start_date = dt.datetime.now() - dt.timedelta(int(365.25 * num_of_years))
    start_date_str = dt.datetime.strftime(start_date, "%Y-%m-%d")
    end_date = dt.datetime.now()
    end_date_str = dt.datetime.strftime(end_date, "%Y-%m-%d")
    # print(start_date)
    # print(end_date)
    df = retrieve_ohlcv_from_to(
        engine,
        exchange_code=exchange_code,
        ticker=ticker,
        start_date=start_date_str,
        end_date=end_date_str,
    )
    print(f"\nInfo: {df.info()}")
    print(df.head(3))
    # print(df.tail(3))
    df.columns = ["Date", "Open", "High", "Low", "Close", "Volume"]
    print(df.info())
    return df


In [5]:
df = localData(exchange="XNAS", ticker="GOOG", years=1)

Exchange_id is: 2
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1879 entries, 2018-10-26 to 2026-04-21
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   date    1879 non-null   object 
 1   open    1879 non-null   float64
 2   high    1879 non-null   float64
 3   low     1879 non-null   float64
 4   close   1879 non-null   float64
 5   volume  1879 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 102.8+ KB

Info: None
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1879 entries, 2018-10-26 to 2026-04-21
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Date    1879 non-null   object 
 1   Open    1879 non-null   float64
 2   High    1879 non-null   float64
 3   Low     1879 non-null   float64
 4   Close   1879 non-null   float64
 5   Volume  1879 non-null   int64  
dtypes: float64(4), int64(1), object(1)
memory usage: 102.8+ KB
No

In [3]:
def SMA(values, n):
    """
    Return simple moving average of `values`, at
    each step taking into account `n` previous values.
    """
    return pd.Series(values).rolling(n).mean()

In [4]:
class SmaCross(Strategy):
    # Define the two MA lags as *class variables*
    # for later optimization
    n1 = 10
    n2 = 20
    
    def init(self):
        # Precompute the two moving averages
        self.sma1 = self.I(SMA, self.data.Close, self.n1)
        self.sma2 = self.I(SMA, self.data.Close, self.n2)
    
    def next(self):
        # If sma1 crosses above sma2, close any existing
        # short trades, and buy the asset
        if crossover(self.sma1, self.sma2):
            self.position.close()
            self.buy()

        # Else, if sma1 crosses below sma2, close any existing
        # long trades, and sell the asset
        elif crossover(self.sma2, self.sma1):
            self.position.close()
            self.sell()

In [5]:
bt = Backtest(localData("XNAS", "GOOG", 10), SmaCross, cash=10_000, commission=.002)
stats = bt.run()
stats

Exchange_id is: 2


Backtest.run:   0%|          | 0/1858 [00:00<?, ?bar/s]

/tmp/ipykernel_27500/2308637473.py:2: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = bt.run()


Start                     2018-10-26 00:00:00
End                       2026-04-20 00:00:00
Duration                   2733 days 00:00:00
Exposure Time [%]                    98.34931
Equity Final [$]                   8162.25723
Equity Peak [$]                   11211.21993
Commissions [$]                    3201.91305
Return [%]                          -18.37743
Buy & Hold Return [%]                555.1549
Return (Ann.) [%]                    -2.68804
Volatility (Ann.) [%]                    29.5
CAGR [%]                             -1.85496
Sharpe Ratio                         -0.09112
Sortino Ratio                        -0.12831
Calmar Ratio                         -0.04639
Alpha [%]                           -87.21077
Beta                                  0.12399
Max. Drawdown [%]                   -57.94979
Avg. Drawdown [%]                    -19.2185
Max. Drawdown Duration     2460 days 00:00:00
Avg. Drawdown Duration      538 days 00:00:00
# Trades                          

In [6]:
bt.plot()

GridPlot(id='p1320', ...)

In [7]:
%%time

stats = bt.optimize(n1=range(5, 30, 5),
                    n2=range(10, 70, 5),
                    maximize='Equity Final [$]',
                    constraint=lambda param: param.n1 < param.n2)
stats

Backtest.optimize:   0%|          | 0/50 [00:00<?, ?it/s]

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/1838 [00:00<?, ?bar/s]

Backtest.run:   0%|          | 0/1863 [00:00<?, ?bar/s]

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of ba

...

/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)
/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1637: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  for stats in (bt.run(**params)


Backtest.run:   0%|          | 0/1843 [00:00<?, ?bar/s]

CPU times: user 371 ms, sys: 248 ms, total: 619 ms
Wall time: 1.14 s


/home/ubuntuuser/miniconda3/envs/analysis/lib/python3.12/site-packages/backtesting/backtesting.py:1545: UserWarning: Some trades remain open at the end of backtest. Use `Backtest(..., finalize_trades=True)` to close them and include them in stats.
  stats = self.run(**dict(zip(heatmap.index.names, best_params)))


Start                     2018-10-26 00:00:00
End                       2026-04-20 00:00:00
Duration                   2733 days 00:00:00
Exposure Time [%]                     95.5804
Equity Final [$]                  30532.16586
Equity Peak [$]                   37131.72586
Commissions [$]                    3176.23046
Return [%]                          205.32166
Buy & Hold Return [%]               559.89199
Return (Ann.) [%]                    16.15752
Volatility (Ann.) [%]                34.84163
CAGR [%]                             10.84031
Sharpe Ratio                          0.46374
Sortino Ratio                           0.821
Calmar Ratio                          0.40584
Alpha [%]                           134.12328
Beta                                  0.12716
Max. Drawdown [%]                   -39.81273
Avg. Drawdown [%]                    -4.54322
Max. Drawdown Duration     1269 days 00:00:00
Avg. Drawdown Duration       47 days 00:00:00
# Trades                          

In [31]:
stats._strategy

<Strategy SmaCross(n1=25,n2=35)>

In [32]:
stats.tail()


SQN                                                         1.217347
Kelly Criterion                                             0.271708
_strategy                                      SmaCross(n1=25,n2=35)
_equity_curve                        Equity  DrawdownPct Drawdown...
_trades                Size  EntryBar  ExitBar  EntryPrice  ExitP...
dtype: object

In [33]:
stats['_equity_curve'] 

,Equity,DrawdownPct,DrawdownDuration
2018-10-26,10000.000000,0.000000,NaT
2018-10-29,10000.000000,0.000000,NaT
2018-10-30,10000.000000,0.000000,NaT
2018-10-31,10000.000000,0.000000,NaT
2018-11-01,10000.000000,0.000000,NaT
...,...,...,...
2026-04-09,32549.345864,0.123409,NaT
2026-04-10,32618.245864,0.121553,NaT
2026-04-13,32248.305864,0.131516,NaT
2026-04-14,31043.085864,0.163974,NaT


In [17]:
stats['_trades']

,Size,EntryBar,ExitBar,EntryPrice,ExitPrice,SL,TP,PnL,Commission,ReturnPct,EntryTime,ExitTime,Duration,Tag,"Entry_SMA(C,25)","Exit_SMA(C,25)","Entry_SMA(C,35)","Exit_SMA(C,35)"
0,191,45,48,52.0500,53.8055,None,None,294.863699,40.436801,0.029660,2019-01-03,2019-01-08,5 days,None,52.17254,52.16542,52.123829,52.262457
1,-190,48,61,53.8055,54.0055,None,None,-78.968180,40.968180,-0.007725,2019-01-08,2019-01-28,20 days,None,52.16542,52.74540,52.262457,52.578914
2,188,61,140,54.0055,57.4245,None,None,600.874320,41.897680,0.059182,2019-01-28,2019-05-21,113 days,None,52.74540,59.79972,52.578914,59.964657
3,-187,140,177,57.4245,57.3430,None,None,-27.682545,42.923045,-0.002578,2019-05-21,2019-07-15,55 days,None,59.79972,55.32810,59.964657,55.016557
4,187,177,216,57.3430,60.2000,None,None,490.297918,43.961082,0.045723,2019-07-15,2019-09-09,56 days,None,55.32810,59.08526,55.016557,59.187829
5,-186,216,223,60.2000,61.3755,None,None,-263.869086,45.226086,-0.023566,2019-09-09,2019-09-18,9 days,None,59.08526,59.74884,59.187829,59.650214
6,179,223,251,61.3755,63.7725,None,None,384.260016,44.802984,0.034977,2019-09-18,2019-10-28,40 days,None,59.74884,61.43822,59.650214,61.449457
7,-178,251,256,63.7725,63.8225,None,None,-54.323820,45.423820,-0.004786,2019-10-28,2019-11-04,7 days,None,61.43822,61.83496,61.449457,61.760671
8,177,256,337,63.8225,69.9710,None,None,1040.921601,47.362899,0.092145,2019-11-04,2020-03-03,120 days,None,61.83496,72.69972,61.760671,72.784100
9,-176,337,375,69.9710,64.8000,None,None,862.656608,47.439392,0.070050,2020-03-03,2020-04-27,55 days,None,72.69972,59.73760,72.784100,59.147629
